# M02 — Prismas gravimétricos y límite de aceptación inversa

**Level 3 · Prerrequisitos: notebook de fundamentos; lectura: 35 minutos.**

## Objetivos

Ejecute el operador directo de prismas SimPEG/Geoana sobre el control sintético congelado y compare con el oráculo Choclo independiente. Falló la puerta nativa conservada de dirección retenida-48; las comprobaciones inversas dependientes están bloqueadas. Este notebook registra ese resultado negativo.

## Preparación

Use el entorno pipeline Python 3.12 ya revisado. Este notebook no instala nada.

Importe el código del repositorio y confirme su archivo fuente.

In [ ]:
import runpy,sys
from pathlib import Path
import numpy as np
root=Path.cwd();sys.path.insert(0,str(root/'data-pipeline'))
import gravity_forward
# Load the product's existing synthetic control, not a field survey.
fixture=runpy.run_path(str(root/'tests/numerics/test_gravity_forward.py'))
print('Motor gravimétrico:',gravity_forward.ENGINE)

## Datos

Ejecute el operador directo de prismas SimPEG/Geoana sobre el control sintético congelado y compare con el oráculo Choclo independiente. Falló la puerta nativa conservada de dirección retenida-48; las comprobaciones inversas dependientes están bloqueadas. Este notebook registra ese resultado negativo.

**Files:** tests/numerics/test_gravity_forward.py, docs/design/features/m02-prism-operator/validation.md

## Método

Para receptor exterior, la aceleración hacia arriba es g_z=G∫_V Δρ(z′−z)/|r−r′|³ dV. El operador discreto d=Jρ usa contraste kg m⁻³, predicción mGal y J en mGal/(kg m⁻³). Masa positiva enterrada da gz hacia arriba negativo. La inversa agrega errores, referencia, cotas y regularización; los modelos de campos potenciales no son únicos.

Ejecute el método del producto sobre el control declarado.

In [ ]:
request=fixture['request']()
# Compare production prediction with a separate physical prism implementation.
result=gravity_forward.forward_gravity(request)
oracle=fixture['choclo_oracle'](request)
print({'prediccion_producto_hacia_arriba_mGal':result['gz_up_mgal'].tolist(),'oraculo_Choclo_mGal':oracle.tolist(),'diferencia_absoluta_maxima_mGal':float(np.max(np.abs(result['gz_up_mgal']-oracle)))})

Control adversarial/negativo: aísle un supuesto o incumpla una restricción física declarada.

In [ ]:
from copy import deepcopy
invalid=deepcopy(request);invalid['receivers_m']=np.array([[0.0,0.0,-50.0]])
try:
    # Closed-mesh interior receivers violate the forward contract.
    gravity_forward.forward_gravity(invalid)
except ValueError as error:
    print('Rechazo geométrico esperado:',str(error))
else:
    raise AssertionError('Interior receiver was accepted')

## Estado del arte

SimPEG/Geoana es motor directo de producción; Choclo es oráculo analítico independiente. Concordancia directa no es recuperación de densidad. La inversa sigue no aceptada porque falló dirección retenida-48; las pruebas de nulo positivo y primer pliegue no corrieron.

## Resultados

Solo se evalúa comportamiento directo sintético acotado. El receptor inválido debe rechazarse. No se afirma inversión, ajuste de campo, resultado retenido ni capacidad del host; conserve la puerta fallida ligada al código.

## Resumen

Un núcleo directo validado puede coexistir con una inversa no resuelta. No convierta un fallo conservado del solver en éxito con gráficas o tolerancias cambiadas.

## Referencias

- https://docs.simpeg.xyz/v0.25.2/content/api/generated/simpeg.potential_fields.gravity.Simulation3DIntegral.html
- https://www.fatiando.org/choclo/latest/api/generated/choclo.prism.gravity_u.html
- https://doi.org/10.1016/j.cageo.2015.09.015